In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.core.guarded_eval import dict_values
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier,LogisticRegression
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import  KNeighborsClassifier




In [27]:
df=pd.read_csv("16-diabetes.csv")
df.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [28]:
df.shape

(768, 9)

In [29]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='str')

In [30]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [31]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [32]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


In the describe table, the minimum values are interesting. No human can have 0 BloodPressure or 0 BMI in real life. I think these zeros may represent missing data. So, I researched this issue and asked ChatGPT about it.

In [33]:
dict_columns={}
for col in df.columns:
    dict_columns[col]=(df[col]==0).sum()

dict_df=pd.DataFrame(list(dict_columns.items()),columns=["Col_Name","Zeros_Count"])
dict_df["Percentage"]=(dict_df["Zeros_Count"]/768)*100
print(dict_df)

                   Col_Name  Zeros_Count  Percentage
0               Pregnancies          111   14.453125
1                   Glucose            5    0.651042
2             BloodPressure           35    4.557292
3             SkinThickness          227   29.557292
4                   Insulin          374   48.697917
5                       BMI           11    1.432292
6  DiabetesPedigreeFunction            0    0.000000
7                       Age            0    0.000000
8                   Outcome          500   65.104167


Okey now we have zeros represent missing data. If I dropped these features, that would be a big mistake.As these features are critical.

In [34]:
columns_to_change=['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'Age']

for col in columns_to_change:
      df.loc[df[col]==0,col]=df[df[col] != 0][col].median()


In [35]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,121.656250,72.386719,29.108073,140.671875,32.455208,0.471876,33.240885,0.348958
std,3.369578,30.438286,12.096642,8.791221,86.383060,6.875177,0.331329,11.760232,0.476951
min,0.000000,44.000000,24.000000,7.000000,14.000000,18.200000,0.078000,21.000000,0.000000
25%,1.000000,99.750000,64.000000,25.000000,121.500000,27.500000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,29.000000,125.000000,32.300000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


###ML

In [36]:
X=df.drop("Outcome",axis=1)
y=df["Outcome"]
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=15)


In [47]:
models={
    "Logistic Classifier":LogisticRegression(),
    "Ridge Classifier":RidgeClassifier(),
    "SVC":SVC(),
    "KNN":KNeighborsClassifier(),
    "Decision Tree":DecisionTreeClassifier(),
    "Random Forest":RandomForestClassifier(),
    "AdaBoost":AdaBoostClassifier()
}


params = {
    "Logistic Classifier": {
        "C": [0.1, 1, 10],
        "penalty":["l2"]
    },

    "Ridge Classifier": {
    "alpha": [0.1, 1, 10]
    },

    "SVC": {
    "C": [0.1, 1, 10],
    "kernel": ["linear", "rbf"]
    },

    "KNN": {
    "n_neighbors": [3,5,7,9],
    "weights": ["uniform", "distance"]
    },

    "Decision Tree": {
    "max_depth": [3,5,10],
    "criterion": ["gini", "entropy"]
    },

    "Random Forest": {
    "n_estimators": [50,100],
    "max_depth": [5,10]
    },

    "AdaBoost": {
    "n_estimators": [50,100],
    "learning_rate": [0.01,0.1,1]
    },


}

In [48]:
for model_name, model in models.items():

    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params[model_name],
        cv=5,
        n_iter=3,
        scoring="accuracy"
    )

    random_search.fit(X_train, y_train)

    print(model_name)
    print(random_search.best_score_)
    print(random_search.best_params_)

C:\Users\CANSU\PycharmProjects\Course_Exercise_Projects\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\CANSU\PycharmProjects\Course_Exercise_Projects\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.htm

Logistic Classifier
0.7777061469265367
{'penalty': 'l2', 'C': 10}
Ridge Classifier
0.777736131934033
{'alpha': 0.1}
SVC
0.7725337331334333
{'kernel': 'linear', 'C': 1}
KNN
0.737856071964018
{'weights': 'distance', 'n_neighbors': 9}
Decision Tree
0.7707796101949025
{'max_depth': 3, 'criterion': 'entropy'}
Random Forest
0.7742128935532234
{'n_estimators': 50, 'max_depth': 10}
AdaBoost
0.7602848575712144
{'n_estimators': 100, 'learning_rate': 1}
